# Ch.3 — Autoencoders for Anomaly Detection

> Train a neural network to reconstruct normal transactions. Fraud = high reconstruction error, because the bottleneck never learned to encode fraud patterns.

**Dataset:** Credit Card Fraud Detection — 284,807 transactions, 0.17% fraud  
**Task:** Learn normal transaction manifold, flag reconstruction errors, target ~78% recall @ 0.5% FPR

## The Core Idea

An autoencoder compresses input through a bottleneck and reconstructs it:

$$\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 = \|\mathbf{x} - g_\phi(f_\theta(\mathbf{x}))\|^2$$

Trained **only on normal data**, it becomes an expert at reconstructing legitimate transactions. Fraud transactions — never seen during training — produce high reconstruction error.

| Component | Dimension | Role |
|-----------|-----------|------|
| Input $\mathbf{x}$ | 30 | Transaction features |
| Bottleneck $\mathbf{z}$ | 7-14 | Compressed representation |
| Output $\hat{\mathbf{x}}$ | 30 | Reconstructed transaction |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
IMG_DIR = "img/"

import os
os.makedirs(IMG_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ── Load, split, and scale ─────────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")
X = df.drop("Class", axis=1).values
y = df["Class"].values
feature_names = [c for c in df.columns if c != "Class"]

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# CRITICAL: Train only on normal data
X_normal = X_train[y_train == 0]
print(f"Normal training samples: {len(X_normal):,}")
print(f"Test samples: {len(X_test):,} ({y_test.sum()} fraud)")

# Scale features
scaler = StandardScaler()
X_normal_s = scaler.fit_transform(X_normal)
X_test_s = scaler.transform(X_test)

In [ ]:
# ── Define the Autoencoder ─────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim=30, bottleneck=7):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 20), nn.ReLU(),
            nn.Linear(20, 14), nn.ReLU(),
            nn.Linear(14, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 14), nn.ReLU(),
            nn.Linear(14, 20), nn.ReLU(),
            nn.Linear(20, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = Autoencoder(input_dim=X_normal_s.shape[1], bottleneck=7)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# ── Train the Autoencoder ──────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_normal_s)),
    batch_size=256, shuffle=True,
)

train_losses = []
n_epochs = 50

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for (batch,) in train_loader:
        x_hat = model(batch)
        loss = criterion(x_hat, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.6f}")

print(f"Final loss: {train_losses[-1]:.6f}")

In [ ]:
# ── Training loss curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, color="#8e44ad", linewidth=2)
ax.set(title="Autoencoder Training Loss", xlabel="Epoch", ylabel="MSE Loss")
ax.axhline(train_losses[-1], color="#e74c3c", linestyle="--", alpha=0.5,
           label=f"Final: {train_losses[-1]:.6f}")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Score: reconstruction error ────────────────────────────────────────────
model.eval()
with torch.no_grad():
    x_test_t = torch.FloatTensor(X_test_s)
    x_hat = model(x_test_t)
    recon_error = ((x_test_t - x_hat) ** 2).mean(dim=1).numpy()

fraud_mask = y_test == 1
print(f"Reconstruction error:")
print(f"  Normal — mean: {recon_error[~fraud_mask].mean():.4f}, std: {recon_error[~fraud_mask].std():.4f}")
print(f"  Fraud  — mean: {recon_error[fraud_mask].mean():.4f}, std: {recon_error[fraud_mask].std():.4f}")
print(f"  Ratio (fraud/normal): {recon_error[fraud_mask].mean() / recon_error[~fraud_mask].mean():.1f}x")

In [ ]:
# ── ROC Curve and Recall @ 0.5% FPR ────────────────────────────────────────
fpr_ae, tpr_ae, thresh_ae = roc_curve(y_test, recon_error)
auc_ae = auc(fpr_ae, tpr_ae)

idx_005 = np.where(fpr_ae <= 0.005)[0][-1]
recall_ae = tpr_ae[idx_005]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr_ae, tpr_ae, color="#8e44ad", linewidth=2, label=f"AE (AUC={auc_ae:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].scatter([fpr_ae[idx_005]], [tpr_ae[idx_005]], color="#e74c3c", s=100, zorder=5,
               label=f"@ 0.5% FPR: {recall_ae:.1%}")
axes[0].set(title="ROC — Autoencoder", xlabel="FPR", ylabel="Recall")
axes[0].legend()

# Reconstruction error distribution
axes[1].hist(recon_error[~fraud_mask], bins=100, alpha=0.6, color="#27ae60",
            label="Legitimate", density=True)
axes[1].hist(recon_error[fraud_mask], bins=50, alpha=0.7, color="#e74c3c",
            label="Fraud", density=True)
axes[1].set(title="Reconstruction Error Distribution", xlabel="MSE", ylabel="Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_autoencoder.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nRecall @ 0.5% FPR: {recall_ae:.2%}")

## Per-Feature Reconstruction Error

Which features does the autoencoder struggle to reconstruct for fraud? This gives interpretability — we can say *which features* look anomalous.

In [ ]:
# ── Per-feature reconstruction error ───────────────────────────────────────
with torch.no_grad():
    per_feat_error = ((x_test_t - x_hat) ** 2).numpy()

# Average per-feature error for fraud vs normal
fraud_feat_err = per_feat_error[fraud_mask].mean(axis=0)
normal_feat_err = per_feat_error[~fraud_mask].mean(axis=0)
ratio = fraud_feat_err / (normal_feat_err + 1e-8)

top_idx = np.argsort(ratio)[-10:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([feature_names[i] for i in top_idx], ratio[top_idx], color="#8e44ad")
ax.set(title="Per-Feature Reconstruction Error Ratio (Fraud / Normal)",
       xlabel="Error Ratio")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5, label="Ratio = 1 (no difference)")
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}per_feature_error.png", dpi=150, bbox_inches="tight")
plt.show()

## Bottleneck Dimension Sweep

The bottleneck dimension $d_z$ is the most critical hyperparameter. Too wide → fraud also reconstructs well. Too narrow → nothing reconstructs well.

In [ ]:
# ── Bottleneck dimension sweep ─────────────────────────────────────────────
bottleneck_dims = [3, 5, 7, 10, 14, 20, 28]
results_bn = []

for d_z in bottleneck_dims:
    torch.manual_seed(SEED)
    ae = Autoencoder(input_dim=X_normal_s.shape[1], bottleneck=d_z)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    crit = nn.MSELoss()

    loader = DataLoader(TensorDataset(torch.FloatTensor(X_normal_s)),
                       batch_size=256, shuffle=True)
    for epoch in range(30):  # fewer epochs for sweep
        ae.train()
        for (batch,) in loader:
            loss = crit(ae(batch), batch)
            opt.zero_grad(); loss.backward(); opt.step()

    ae.eval()
    with torch.no_grad():
        err = ((x_test_t - ae(x_test_t)) ** 2).mean(dim=1).numpy()
    fpr_i, tpr_i, _ = roc_curve(y_test, err)
    idx = np.where(fpr_i <= 0.005)[0][-1]
    results_bn.append({"bottleneck": d_z, "AUC": auc(fpr_i, tpr_i),
                       "Recall@0.5%FPR": tpr_i[idx]})
    print(f"  d_z={d_z:2d}: Recall@0.5%FPR = {tpr_i[idx]:.2%}")

bn_df = pd.DataFrame(results_bn)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bn_df["bottleneck"], bn_df["Recall@0.5%FPR"], "o-", color="#8e44ad", linewidth=2)
ax.set(title="Bottleneck Dimension vs Recall", xlabel="Bottleneck Dim (d_z)",
       ylabel="Recall @ 0.5% FPR")
ax.axhline(0.80, color="#e74c3c", linestyle="--", alpha=0.5, label="Target: 80%")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}bottleneck_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## Progress Check

| Constraint | Target | Status |
|------------|--------|--------|
| #1 DETECTION | >80% recall | ❌ ~78% — very close! |
| #2 PRECISION | <0.5% FPR | ✅ ROC thresholding |
| #3 REAL-TIME | <100ms | ✅ ~10ms inference |
| #4 ADAPTABILITY | Drift handling | ❌ Static model |
| #5 EXPLAINABILITY | Justifiable | ⚡ Per-feature error gives partial explanation |

**Next**: Ch.4 — One-Class SVM (boundary-based, complementary signal for ensemble)

## Exercises

In [ ]:
# ── Exercise 1: Denoising Autoencoder ──────────────────────────────────────
# TODO: Add Gaussian noise (σ=0.3) to inputs during training
# but still reconstruct the CLEAN input
# Compare recall vs standard autoencoder
# Hint: noisy_batch = batch + torch.randn_like(batch) * 0.3
#        loss = criterion(model(noisy_batch), batch)  # reconstruct clean!

pass

In [ ]:
# ── Exercise 2: Dropout regularization ─────────────────────────────────────
# TODO: Add nn.Dropout(0.2) after each ReLU in the encoder
# Does dropout improve generalization (lower FPR on normal test data)?
# Compare ROC curves with and without dropout

pass

In [ ]:
# ── Exercise 3: Contaminated training ──────────────────────────────────────
# TODO: Train the autoencoder on ALL training data (including fraud)
# Compare recall vs training on normal-only
# With 0.17% contamination, how much does performance degrade?

pass